# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields according to best FAIR data practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and display high-level information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name if hasattr(metadata, 'name') else ''}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else ''}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else ''}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else ''}")

## 2. Data Overview
Review available record sets, their fields, columns and `@id`s.

In [ ]:
# List record sets and their fields by @id
record_sets = list(dataset.metadata.recordSets) if hasattr(dataset.metadata, 'recordSets') else []
if not record_sets:
    # fallback: try .recordSet field
    record_sets = list(getattr(dataset.metadata, 'recordSet', []))

if not record_sets:
    print("No record sets found.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"Record set @id: {rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)}")
        print(f"  Name: {rs.get('name', getattr(rs, 'name', ''))}")
        print(f"  Description: {rs.get('description', getattr(rs, 'description', ''))}")
        print("  Fields and Columns:")
        # Fields may be 'fields' or 'field' or 'columns', by Croissant spec
        fields = rs.get('fields', getattr(rs, 'fields', [])) or rs.get('field', getattr(rs, 'field', []))
        columns = rs.get('columns', getattr(rs, 'columns', [])) or rs.get('column', getattr(rs, 'column', []))
        if fields:
            for f in fields:
                print(f"    Field @id: {f['@id'] if isinstance(f, dict) else getattr(f, '@id', None)}  Name: {f.get('name', getattr(f, 'name', ''))}")
        if columns:
            for c in columns:
                print(f"    Column @id: {c['@id'] if isinstance(c, dict) else getattr(c, '@id', None)}  Name: {c.get('name', getattr(c, 'name', ''))}")
        print("")

    # If you want to see examples from one RecordSet, uncomment below:
    chosen_record_set_id = record_sets[0]['@id'] if isinstance(record_sets[0], dict) else getattr(record_sets[0], '@id', None)
    print(f"\nFirst RecordSet @id: {chosen_record_set_id}, sample records:")
    for i, x in enumerate(dataset.records(record_set=chosen_record_set_id)):
        print(x)
        if i >= 2:
            break

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use Croissant `@id`s.

In [ ]:
# Extract data from each record set by @id.
# Edit these if more record sets are present (see previous cell's output)
from collections.abc import Iterable

def get_id(rs):
    if isinstance(rs, dict):
        return rs.get('@id', None)
    return getattr(rs, '@id', None)

record_set_ids = [get_id(rs) for rs in record_sets if get_id(rs)] if record_sets else []
print(f"Record set @id(s): {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))  # Each record is a dict
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and categorizing/grouping by key attributes, using `@id` references for fields.

In [ ]:
# For demonstration, pick a record set and numeric field by @id (edit as needed)
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    df = dataframes[example_record_set_id]
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    # Try to select a numeric field
    numeric_col = None
    for col in df.columns:
        # Try to infer numeric field by dtype or by name
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break
        # else, try to parse column to float
        try:
            pd.to_numeric(df[col])
            numeric_col = col
            break
        except Exception:
            continue
    if numeric_col:
        df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')
        threshold = df[numeric_col].mean() if df[numeric_col].mean() is not None else 0
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records with {numeric_col} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"Normalized {numeric_col} for filtered records:")
        print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())
        # Try group by another field if available
        group_field = None
        for col in df.columns:
            if col != numeric_col and df[col].nunique() < 8:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_col].mean().to_frame()
            print(f"Grouped mean {numeric_col} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No obvious numeric field found for EDA.")
else:
    print("No record set to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. (Matplotlib and seaborn can be used.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_col:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_col].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_col} in RecordSet {example_record_set_id}")
    plt.xlabel(numeric_col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
    # If grouped_df exists, plot bar
    if 'grouped_df' in locals() and group_field:
        grouped_df.plot(kind='bar', legend=False, figsize=(8,5))
        plt.title(f"Mean {numeric_col} by {group_field}")
        plt.ylabel(f"Mean {numeric_col}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field or record set found for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset's Croissant package for mass spectrometry protein analysis using the `mlcroissant` library.

- Dataset metadata and structure (record sets, fields) were summarized by referencing all entities by their Croissant `@id`.
- Data was loaded dynamically from each record set into pandas DataFrames.
- Numeric fields were filtered, normalized, and visualized, demonstrating typical data processing steps.

This workflow provides a FAIR-compliant approach to dataset exploration, and can be adapted for downstream analysis or machine learning workflows.
